# safety: limits, workspace, e-stop

Library notebook for `gambit_arm`. Every joint command passes through `validate_command`
before it reaches a backend (real servos, Gazebo, LeRobot). Limits come from
`gambit_bringup/config/safety.yaml`; the same file is meant to drive the C++ daemon and the
STM32 firmware (`firmware/stm32/Core/Src/limits.c`), which enforces them again in real time.

Milestone: *Hardware and execution* (safe operation is a precondition for every trial).

**TODO**
- [ ] load safety.yaml, joint limits with margin, clamp and validate joint commands
- [ ] velocity / step-size check between consecutive commands
- [ ] workspace box and tilt check on the FK tool pose
- [ ] latched software e-stop with heartbeat timeout and over-current hold time
- [ ] self-collision / table-collision check along the whole segment, not just endpoints
      (sample FK between waypoints, check every intermediate frame's height)
- [ ] measure real limits (safety.yaml says "measure/verify") and tighten them
- [ ] wire `EStop` to the servo bus: on trip, hold position or disable torque per `estop.on_trip`
- [ ] a physical e-stop button (cuts servo power); software e-stop is not a substitute
- [ ] write the self-checks at the bottom as asserts

In [ ]:
import time
from dataclasses import dataclass, field
import numpy as np

%run ./kinematics.ipynb

SAFETY = load_yaml("safety.yaml")
list(SAFETY)

## Joint limits

URDF limits (copied into safety.yaml), shrunk by `joint_margin` so we never drive a servo
into its end stop. `clamp_joints` is for gentle use (teleop, small corrections);
`validate_command` is the gatekeeper that *rejects* bad commands instead of silently fixing them.

In [ ]:
def joint_limits(cfg=SAFETY, names=ARM_JOINTS):
    """(lower, upper) arrays for `names`, with the margin applied."""
    # Design: read joints.<name>.lower/upper from safety.yaml and shrink both ends by
    # joint_margin (lower + m, upper - m). One function so every caller agrees on the margin.
    # TODO: implement
    raise NotImplementedError

def clamp_joints(q, cfg=SAFETY, names=ARM_JOINTS) -> np.ndarray:
    """q clipped into the margin-shrunk limits (for teleop / small corrections only)."""
    # Design: np.clip against joint_limits. Never use this to "fix" planner output: a planner
    # that produces out-of-limit targets has a bug, and validate_command should reject it.
    # TODO: implement
    raise NotImplementedError

## Workspace

The gripper tip must stay inside the axis-aligned box in `workspace`, and the approach must
stay within `max_tilt_deg` of vertical when the tip is low (near pieces). Higher up (transit,
park) tilt is not checked.

In [ ]:
def in_workspace(p, cfg=SAFETY) -> bool:
    """True if point p (x, y, z) is inside the workspace box (inclusive)."""
    # Design: compare each coordinate with workspace.x / .y / .z [lo, hi] from safety.yaml.
    # TODO: implement
    raise NotImplementedError

def check_pose(q, cfg=SAFETY, low_z=None) -> list:
    """List of problems with the FK tool pose of q (empty = fine)."""
    # Design:
    # - FK once; check the tip against in_workspace.
    # - Tilt is only checked when the tip is "low" (below board top + hover height by
    #   default): near pieces a tilted gripper knocks neighbors over; in transit it does not matter.
    # - Return human-readable strings (with the numbers) rather than a bool: they go straight
    #   into error messages and logs.
    # TODO: implement
    raise NotImplementedError

## Validating a command

Checks, in order: shape and finiteness, joint limits, step size and velocity versus the
previous command (if given), then the FK pose. Returns a list of human-readable problems.
`assert_safe` raises, for code paths that must stop.

In [ ]:
class SafetyViolation(RuntimeError):
    pass

def validate_command(q, q_prev=None, dt=None, cfg=SAFETY, check_fk=True) -> list:
    """All problems with joint command q (empty list = safe to send).
    q_prev / dt: previous command and time between them, for step and velocity checks."""
    # Design (checks in order, collecting every problem rather than stopping at the first):
    # 1. shape (len(ARM_JOINTS),) and all finite; if not, return immediately (nothing else
    #    is meaningful on NaNs).
    # 2. each joint inside joint_limits (name the joint in the message).
    # 3. with q_prev: max |q - q_prev| <= velocity.max_step_rad; with dt too: max step/dt <=
    #    velocity.max_joint_rad_s.
    # 4. if check_fk: check_pose(q).
    # Message prefixes ("shape", "joint <name>", "step", "velocity", "workspace", "tilt")
    # are what tests and logs grep for, so keep them stable.
    # TODO: implement
    raise NotImplementedError

def assert_safe(q, **kw) -> np.ndarray:
    """validate_command, raising SafetyViolation (all problems joined) if anything is wrong."""
    # TODO: implement
    raise NotImplementedError

def gripper_command_ok(g: float, cfg=SAFETY) -> bool:
    """Gripper target within joints.gripper limits."""
    # TODO: implement (1 line)
    raise NotImplementedError

## E-stop

A latched state machine: any trip stays tripped until `clear()` (if `require_manual_reset`).
`feed_heartbeat()` must be called by the host loop; `check()` trips if it goes stale. Servo
telemetry (current, temperature) trips only after staying over the limit for
`overcurrent_hold_s`, so a short spike while grasping does not stop the game.

In [ ]:
@dataclass
class EStop:
    cfg: dict = field(default_factory=lambda: SAFETY)
    clock: object = time.monotonic
    tripped: bool = False
    reasons: list = field(default_factory=list)
    _last_heartbeat: float = None
    _over_since: dict = field(default_factory=dict)

    def trip(self, reason: str) -> bool:
        """Latch the e-stop. `reason` = "<cause>: details", cause from estop.trip_on in safety.yaml."""
        # Design: set tripped, append the reason (keep all reasons: the first one is usually
        # the real cause, later ones are consequences).
        # TODO: implement
        raise NotImplementedError

    def clear(self):
        """Operator reset. TODO: re-read servo state and refuse to clear while a cause persists."""
        # Design: reset tripped / reasons / over-limit timers and restart the heartbeat clock
        # (otherwise the very next check() trips on a stale heartbeat).
        # TODO: implement
        raise NotImplementedError

    def feed_heartbeat(self):
        """Host loop calls this every cycle."""
        # TODO: implement (store self.clock())
        raise NotImplementedError

    def check(self) -> bool:
        """Call every control cycle. Returns True if motion is allowed."""
        # Design: trip with "heartbeat_lost: ..." if a heartbeat was ever fed and the last one
        # is older than estop.heartbeat_timeout_s; return not tripped. Use self.clock (injected)
        # so tests can drive time with a fake clock.
        # TODO: implement
        raise NotImplementedError

    def update_telemetry(self, currents_ma=None, temps_c=None) -> bool:
        """currents_ma / temps_c: {joint: value}. Trips on sustained over-current or over-temperature."""
        # Design:
        # - Over-current must persist for torque.overcurrent_hold_s before tripping: remember
        #   per joint when it first went over (_over_since), forget it as soon as it drops back.
        #   Reason: grasping causes short current spikes that are not faults.
        # - Over-temperature trips immediately (it changes slowly, so one reading is real).
        # - Compare |current| (sign depends on direction).
        # TODO: implement
        raise NotImplementedError

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - joint_limits: lower < upper everywhere; upper[pan] == 1.91986 - joint_margin
# - clamp_joints([5, -5, 0, 0, 0]) pins the first two joints to their limits
# - top-down IK at e4 hover -> validate_command returns []
# - pan = 3.0 -> a "joint shoulder_pan" problem; q vs q_prev + 0.5 -> a "step" problem;
#   q_prev + 0.1 with dt = 0.01 -> "velocity"; NaNs -> first problem starts with "shape"
# - hover over a1 and h8 and graveyard slot 3 (+1 cm) are in_workspace; (0, 0, 0.5) is not
# - assert_safe on a bad command raises SafetyViolation
# - gripper_open and gripper_closed from board.yaml pass gripper_command_ok
# - EStop with a fake clock (_t = [0.0]; EStop(clock=lambda: _t[0])):
#     over-current for 0.1 s -> still OK; for 0.4 s -> tripped; stays tripped after current
#     drops (latched) with "overcurrent" in reasons; clear() -> OK; no heartbeat for 1.5 s ->
#     tripped with reason starting "heartbeat_lost"
# Example shape of a test:
# assert any(p.startswith("step") for p in validate_command(q_e4, q_prev=q_e4 + 0.5))
print("Self-check: not written yet (see plan above)")
print("safety OK")